1단계: PDF 텍스트 추출
- 라이브러리 설치: !pip install pdfplumber

In [1]:
# PDF 텍스트 추출 라이브러리 import
import pdfplumber

# PDF 파일 열기
with pdfplumber.open(path_or_fp="sample_korean_long.pdf") as pdf:
    text = ""
    for page in pdf.pages:
        text += page.extract_text() + "\n"

# 텍스트 추출 확인
print("총 텍스트 길이:", len(text))
print(text[:500]) # 앞부분만 확인

총 텍스트 길이: 906
제목: 인공지능 개요
인공지능은 현대 사회에서 다양한 분야에 적용되고 있습니다.
의료, 금융, 제조업, 교육 등 거의 모든 산업에서 인공지능 기술이 활용되고 있으며,
특히 대규모 언어 모델(LLM)은 방대한 텍스트 데이터를 학습하여
사람과 유사한 방식으로 질문에 답변할 수 있습니다.
이러한 모델은 단순한 질의응답을 넘어, 문서 요약, 번역, 창의적 글쓰기 등
다양한 작업을 수행할 수 있어 실무에서 큰 가치를 제공합니다.
제목: 벡터 데이터베이스
벡터 데이터베이스는 인공지능과 함께 사용될 때 강력한 도구가 됩니다.
문서를 벡터로 변환하여 저장하면, 질문과 가장 유사한 문서를 빠르게 검색할 수 있습니다.
이는 단순 키워드 검색보다 훨씬 정교한 결과를 제공합니다.
Qdrant, Pinecone, Weaviate 등 다양한 벡터 데이터베이스가 존재하며,
특히 Qdrant는 고성능 검색과 인덱싱을 지원하여 RAG 시스템 구축에 널리 활용됩니다.
제목: RAG 시스템
RAG(Retrieval-


2단계: 텍스트 청크 분할
- 문단 단위 대신 토큰/문자 기준으로 일정 길이로 나누어 문맥 손실을 최소화
- overlap을 주어 청크 사이 문맥을 이어준다

In [ ]:
# transformers AutoTokenizer 라이브러리 import
from transformers import AutoTokenizer

# AutoTokenizer 모델: 토큰 개수는 모델의 토큰 사전(vocabulary)과 토큰화 규칙에 따라 달라짐
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 토큰 함수 생성
def chunk_text_by_tokens(text, max_tokens=256, overlap=50):
    # 텍스트를 토큰 ID 리스트로 변환
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    # 두번째 루프 전에 206(256-50) step으로 증가 적용, 0번째 0:256, 1번째 206:462
    for i in range(0, len(tokens), max_tokens - overlap):
        chunk = tokens[i:i + max_tokens] # 토큰 범위 지정
        chunks.append(tokenizer.decode(chunk)) # 토큰을 다시 문자열로 변환
    return chunks

chunks = chunk_text_by_tokens(text, max_tokens=256, overlap=50)
print(len(chunks), "개의 청크 생성")

# 토큰 개수 확인
tokens = tokenizer.encode(text, add_special_tokens=False)
print("토큰 개수:", len(tokens))

2 개의 청크 생성
토큰 개수: 399


3단계: 임베딩 단계
- SentenceTransformer 같은 모델을 사용해 각 청크를 고차원 벡터(embedding)로 변환
- 이렇게 얻은 벡터는 의미적 유사성을 계산할 수 있어서, 검색(RAG), 분류, 추천 등에 활용

In [ ]:
# SentenceTransformer 임베딩 라이브러리 import
from sentence_transformers import SentenceTransformer

# 다국어 지원 SentenceTransformer 모델 로드
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 미리 만든 청크 리스트를 임베딩으로 변환
embeddings = model.encode(chunks) # numpy.ndarray 형태의 베터 리스트, 각 벡터는 보통 384 차원

# 결과 확인
print("임베딩 벡터 개수:", len(embeddings)) # 청크 개수와 동일
print("첫번째 청크 임베딩 벡터 크기:", len(embeddings[0])) # 일반적으로 384 차원

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


임베딩 벡터 개수: 2
첫번째 청크 임베딩 벡터 크기: 384
